# 03 Alignment and Analysis

This notebook aligns period embeddings and computes semantic-shift metrics.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.alignment import align_models_to_base
from src.metrics import cosine_shift_from_base, nearest_neighbors, neighbor_shift_from_base, summarize_group_shift
from src.train_embeddings import load_models

In [ ]:
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "data" / "processed"

target_words = ["cloud", "virus", "web", "tablet", "stream", "token", "model", "network", "apple", "platform"]
control_words = ["water", "table", "city", "mother", "winter", "animal"]
all_words = target_words + control_words

In [ ]:
models = load_models(MODEL_DIR)
aligned_models = align_models_to_base(
    models,
    base_period=sorted(models)[0],
    min_anchor_count=20,
    exclude_anchor_words=set(all_words),
)
list(aligned_models)

In [ ]:
cosine_shift = cosine_shift_from_base(aligned_models, all_words)
neighbor_shift = neighbor_shift_from_base(aligned_models, all_words, topn=10)
neighbors = nearest_neighbors(aligned_models, target_words, topn=10)
group_shift = summarize_group_shift(cosine_shift, target_words, control_words)

cosine_shift.head()

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
cosine_shift.to_csv(RESULTS_DIR / "cosine_shift.csv", index=False)
neighbor_shift.to_csv(RESULTS_DIR / "neighbor_shift.csv", index=False)
neighbors.to_csv(RESULTS_DIR / "nearest_neighbors.csv", index=False)
group_shift.to_csv(RESULTS_DIR / "group_shift.csv", index=False)

In [ ]:
neighbors.pivot(index="word", columns="period", values="neighbors")